# Stage 2 - (Region bridge + Projection-A) Regional Alignment Training

Trains the MLP Projection-A to align CLIP ViT-L or DINOv2 ViT-L region-level image features into Qwen 2.5 7B's word embedding space, using the RefCOCO, RefCOCO+, RefCOCOg and Visual Genome training datasets.

Alignment Training for Projection head A: Autoregressive next-token region-level caption prediction through teacher forcing
Loss: Autoregressive cross-entropy over caption tokens + Image-Text Contrastive (ITC) loss.  
Image region token positions are masked with -100 so they contribute zero gradient.

Output: A trained projection_head_a.



## Environment setup

In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


In [ ]:
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HOME'] = str(Path(os.environ['REVA_HF_CACHE_ROOT']).expanduser())
os.environ['TRANSFORMERS_CACHE'] = str(Path(os.environ['REVA_HF_CACHE_ROOT']).expanduser() / 'hub')

import random
import logging
import torch
import glob

from reva.config import ProjectionAConfig
from reva.models import (load_frozen_vit, load_frozen_qwen, build_region_feature_extractor, load_grounding_dino, load_region_feature_extractor, load_ram)
from reva.dataset import (load_coco_detection_samples, load_refcoco_samples, load_visual_genome_samples, load_grit_samples,
                        collect_vqav2_testdev_paths, collect_gqa_testdev_paths, generate_curriculum_decontamination_log, filter_curriculum_from_log)
                        
from reva.training import train_projection_a
from reva.inference import sanity_check_region_alignment, interactive_projection_a_test, region_projection_a_inference

from pathlib import Path

logging.basicConfig(level=logging.INFO)
print('Imports complete.')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Initialize Training Configuration for region bridge (with Projection Head A)

In [ ]:
config = ProjectionAConfig()
config.checkpoint_dir.mkdir(parents=True, exist_ok=True)
config.data_dir.mkdir(parents=True, exist_ok=True)
print(f'Device: {config.device}')
print(f'Level indices: {config.level_indices}')
print(f'Spatial scale: {config.spatial_scale:.4f}')
print(f'Fused dim: {config.fused_dim}  (= {config.num_levels} levels x {config.coordconv_dim})')
print(f'RoI output: {config.roi_output_size}x{config.roi_output_size} = {config.roi_output_size**2} positions')

## Load frozen models and region bridge

In [ ]:
frozen_vit, clip_image_processor = load_frozen_vit(config)

frozen_qwen, qwen_tokenizer = load_frozen_qwen(config)

frozen_qwen.train() # gradient_checkpointing requires train() mode even for a frozen model

In [ ]:
region_extractor = build_region_feature_extractor(config)

## Load training data

In [ ]:
'''
# COCO train2017 images + annotations
mkdir -p /workspace/region_data/coco/annotations
cd /workspace/region_data/coco

wget http://images.cocodataset.org/zips/train2017.zip
unzip train2017.zip
rm train2017.zip

wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
unzip annotations_trainval2017.zip
rm annotations_trainval2017.zip

# COCO train2014 (for RefCOCO)
wget http://images.cocodataset.org/zips/train2014.zip
unzip train2014.zip
rm train2014.zip

# VG images
mkdir -p /workspace/region_data/visual_genome
cd /workspace/region_data/visual_genome

wget https://cs.stanford.edu/people/rak248/VG_100K_2/images.zip
unzip images.zip -d VG_100K
rm images.zip

wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip
unzip images2.zip -d VG_100K_2
rm images2.zip

wget https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/region_descriptions.json.zip
unzip region_descriptions.json.zip
rm region_descriptions.json.zip
'''

coco_samples = load_coco_detection_samples(config.data_dir)
refcoco_samples = load_refcoco_samples(config.data_dir, splits=['train'])
vg_samples = load_visual_genome_samples(config.data_dir, max_per_image=config.vg_max_annotations_per_image)
grit_samples = load_grit_samples(
    config.data_dir, shard=0,
    max_images=250000,        # cap to 250K filtered rows (~125-175K images after dead URLs)
    max_boxes_per_image=16,
    use_ref_exps=True,        # richer targets; set False for terser noun-chunk naming
    min_clip_l14=0.30,        # cleaner image-text alignment
    min_box_frac=0.05,        # box must span >=1 ViT grid cell (24x24) -> avoids degenerate region features
)

# GRIT carries open-vocabulary phrases (beyond COCO/VG), so it joins the rich-description VG bucket
vg_samples = vg_samples + grit_samples

print(f'COCO: {len(coco_samples):,} | RefCOCO: {len(refcoco_samples):,} | VG+GRIT: {len(vg_samples):,} (GRIT: {len(grit_samples):,})')
all_curriculum = coco_samples + refcoco_samples + vg_samples
print("Combined dataset created.")

## Decontamination

In [ ]:
import pickle
from pathlib import Path
from collections import Counter

DECONTAM_DIR = Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination'))

# Prefer the most decontaminated pickle that exists on disk
CANDIDATES = [
    DECONTAM_DIR / 'curriculum_pope_vqav2_mmbencheval_seed_clean.pkl',   # POPE+VQAv2+MMBench+SEED
    DECONTAM_DIR / 'curriculum_pope_vqav2_mmbencheval_clean.pkl',       # POPE+VQAv2+MMBench
    DECONTAM_DIR / 'curriculum_pope_vqav2_clean.pkl',
    DECONTAM_DIR / 'curriculum_pope_clean.pkl',
]

clean_pkl = next(p for p in CANDIDATES if p.exists())
print(f'Loading curriculum from: {clean_pkl}')

with open(clean_pkl, 'rb') as f:
    all_curriculum = pickle.load(f)

# Split for train_projection_a (GRIT stays in the VG bucket, same as before)
coco_samples = [s for s in all_curriculum if s.get('source') == 'coco']
refcoco_samples = [s for s in all_curriculum if s.get('source') in ('refcoco', 'refcocog', 'refcoco+')]
vg_samples = [s for s in all_curriculum if s.get('source') in ('visual_genome', 'grit')]

print(f'Total rows: {len(all_curriculum):,}')
print(f'Unique images: {len({s["image_path"] for s in all_curriculum}):,}')
print(f'COCO: {len(coco_samples):,}')
print(f'RefCOCO variants: {len(refcoco_samples):,}')
print(f'VG + GRIT: {len(vg_samples):,}')
print('Sources:', Counter(s.get('source', 'unknown') for s in all_curriculum))

## Train

In [ ]:
region_extractor, training_log = train_projection_a(
    region_extractor=region_extractor,
    frozen_vit=frozen_vit,
    frozen_qwen=frozen_qwen,
    coco_samples=coco_samples,
    refcoco_samples=refcoco_samples,
    vg_samples=vg_samples,
    clip_image_processor=clip_image_processor,
    qwen_tokenizer=qwen_tokenizer,
    config=config,
    save_every_steps=500,
    resume_from_checkpoint=None,  # fresh start recommended after RPN removal
)

## Load trained region bridge

In [ ]:
config = ProjectionAConfig() 

frozen_vit, clip_image_processor = load_frozen_vit(config)
frozen_qwen, qwen_tokenizer = load_frozen_qwen(config)

region_extractor = build_region_feature_extractor(config)

region_extractor_path = os.environ.get("REVA_PROJECTION_A_WEIGHTS", "projection_a_curriculum_best_weights.pt")

region_extractor = load_region_feature_extractor(region_extractor, region_extractor_path, config)

'''
mkdir -p "${REVA_GROUNDING_DINO_ROOT:-$HOME/reva-data/groundingdino}"
cd "${REVA_GROUNDING_DINO_ROOT:-$HOME/reva-data/groundingdino}"
wget -c https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
wget -c https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py
'''
grounding_dino = load_grounding_dino(config)
ram_proposer = load_ram(config)

## Sanity check

In [ ]:
IMAGES_DIR = Path(os.environ.get("REVA_TEST_IMAGES_ROOT", "my_test_images"))

config.box_source = "ram"

results = sanity_check_region_alignment(
    region_extractor,
    frozen_vit,
    frozen_qwen,
    clip_image_processor,
    qwen_tokenizer,
    test_image_path=str(IMAGES_DIR / "img20.jpg"),
    config=config,
    prompt="Look at this image. Answer in one word or a short phrase.",
    #question="What is on the table?", # comment or use None, if proposals need to be question-independent, (LLM does not filer RAM tags relavent to the question anymore)
    grounding_dino=grounding_dino,
    ram_proposer=ram_proposer,
    box_source="ram", # skips LLM in filtering RAM rags which are question-agnostic, RAM now directly produces tags to G-DINO
    show_boxes=True,
)

In [ ]:
from reva.inference import interactive_projection_a_test
# Press Enter at the box prompt -> hybrid RAM+Qwen+DINO
# Or type x1,y1,x2,y2;x1,y1,x2,y2 -> manual boxes
interactive_projection_a_test(
    frozen_vit, region_extractor, frozen_qwen, qwen_tokenizer,
    clip_image_processor, config,
    grounding_dino=grounding_dino,
    ram_proposer=ram_proposer,
)